In [1]:
# 02_bilstm_baseline.py  (FROZEN)
# BiLSTM puro para eye-tracking ASD vs TD
# - Seeds fixos
# - Descoberta de features se 'features_list.json' não existir
# - Janelação: WINDOW=360, STEP=180
# - Normalização pelos stats do treino
# - Treino com WeightedRandomSampler (balanceia janelas)
# - BCEWithLogitsLoss (pos_weight=1.2)
# - Seleção de limiar no NÍVEL DO SUJEITO (balanced accuracy em VAL)
# - Agregação robusta por sujeito: trimmed mean (10%)
# - Artefatos salvos em pasta datada (modelo, normalização, splits, métricas, matrizes, curvas)
# - NOVO: curvas ROC/PR, curva de calibração, dumps de saídas por janela, métricas estendidas (AUC/AP/threshold)

import os, json, time, csv
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------- Configs principais ----------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

WINDOW, STEP = 360, 180
BATCH = 64
EPOCHS = 12
PATIENCE = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 1.2  # favorece levemente a classe positiva (ASD)
TRIM = 0.10  # 10% mean trimming no voto por sujeito
THRESH_GRID = np.linspace(0.10, 0.90, 81)  # 0.10..0.90 passo 0.01

# Pasta de saída datada
STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
OUT_DIR = f"results_bilstm_debug_{STAMP}"
os.makedirs(OUT_DIR, exist_ok=True)


# ---------------------- Utilidades ----------------------
def log(msg: str):
    print(msg, flush=True)


def to_float_cols(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def mask_valid_windows(X: np.ndarray):
    # remove janelas com NaN/inf
    return np.logical_and(
        np.isfinite(X).all(axis=(1, 2)), ~np.isnan(X).any(axis=(1, 2))
    )


def trimmed_mean(a: np.ndarray, trim: float = TRIM) -> float:
    if len(a) == 0:
        return 0.5
    k = int(len(a) * trim)
    a = np.sort(a)
    if k > 0 and 2 * k < len(a):
        a = a[k:-k]
    return float(a.mean()) if len(a) else 0.5


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def plot_confusion(cm, title, path, labels=("TD", "ASD")):
    plt.figure(figsize=(4.8, 3.8))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels
    )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def robust_read_csv(path, usecols=None, dtype=None):
    """
    Lê CSV de forma tolerante:
    1) tenta engine C (rápido);
    2) cai para engine='python' com on_bad_lines='skip' (tolerante);
    3) se ainda falhar, lê em chunks e concatena.
    """
    # 1) rápido
    try:
        return pd.read_csv(path, low_memory=False, usecols=usecols, dtype=dtype)
    except Exception as e1:
        print(f"[WARN] C-engine falhou ao ler {path}: {e1}")

    # 2) tolerante
    try:
        return pd.read_csv(
            path,
            engine="python",
            sep=",",
            on_bad_lines="skip",  # pula linhas quebradas
            quoting=csv.QUOTE_MINIMAL,
            usecols=usecols,
            dtype=dtype,
        )
    except Exception as e2:
        print(f"[WARN] Python-engine falhou: {e2}")

    # 3) em chunks (último recurso)
    try:
        chunks = []
        for ch in pd.read_csv(
            path,
            engine="python",
            sep=",",
            on_bad_lines="skip",
            quoting=csv.QUOTE_MINIMAL,
            usecols=usecols,
            dtype=dtype,
            chunksize=200_000,
        ):
            chunks.append(ch)
        if not chunks:
            raise RuntimeError("sem chunks válidos")
        return pd.concat(chunks, ignore_index=True)
    except Exception as e3:
        raise RuntimeError(
            f"Falha ao ler CSV mesmo em chunks. Arquivo possivelmente corrompido: {e3}"
        )


# ---------------------- Carregar dados ----------------------
CSV_PATH = "merged_preprocessed_for_model.csv"

# Se existir lista de features, já limitamos a leitura a elas + colunas-chave
features_file = "features_list.json"
features = None
if os.path.exists(features_file):
    with open(features_file, "r", encoding="utf-8") as f:
        features = json.load(f)

base_cols = ["Participant", "RecordingTime [ms]", "Class_numeric"]
usecols = None
dtype_map = None

if features is not None:
    # lê só o necessário e já como float32 para reduzir RAM
    usecols = list(dict.fromkeys(base_cols + features))
    dtype_map = {c: "float32" for c in features}

# leitura robusta
df = robust_read_csv(CSV_PATH, usecols=usecols, dtype=dtype_map)

# Se não havia features_list.json, descobrimos agora (numéricas exceto colunas de controle)
if features is None:
    drop_cols = {"Participant", "Class", "Class_numeric", "RecordingTime [ms]"}
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    features = [c for c in num_cols if c not in drop_cols]

# features
if os.path.exists("features_list.json"):
    with open("features_list.json", "r", encoding="utf-8") as f:
        features = json.load(f)
else:
    drop_cols = {"Participant", "Class", "Class_numeric", "RecordingTime [ms]"}
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    features = [c for c in num_cols if c not in drop_cols]
    # também salvar as features usadas
    save_json(os.path.join(OUT_DIR, "features_list_used.json"), features)

# coerção numérica
df = to_float_cols(df, features + ["Class_numeric"])
df["Class_numeric"] = df["Class_numeric"].astype(int)

participants = df["Participant"].dropna().unique()
classes_per_part = df.groupby("Participant")["Class_numeric"].first()

# ---------------------- Split por sujeito ----------------------
train_parts, test_parts = train_test_split(
    participants,
    test_size=0.2,
    stratify=classes_per_part.loc[participants],
    random_state=SEED,
)
train_parts_final, val_parts = train_test_split(
    train_parts,
    test_size=0.2,
    stratify=classes_per_part.loc[train_parts],
    random_state=SEED,
)

log(
    f"# sujeitos: total={len(participants)} | train={len(train_parts_final)} | val={len(val_parts)} | test={len(test_parts)}"
)

# salvar splits (por reprodutibilidade)
splits = {
    "train_parts": list(map(int, train_parts_final)),
    "val_parts": list(map(int, val_parts)),
    "test_parts": list(map(int, test_parts)),
    "seed": SEED,
    "window": WINDOW,
    "step": STEP,
}
save_json(os.path.join(OUT_DIR, "splits_debug.json"), splits)


# ---------------------- Janelação ----------------------
def generate_windows(df_p, window_size=WINDOW, step=STEP):
    df_p = df_p.sort_values("RecordingTime [ms]")
    data = df_p[features].values.astype(np.float32)
    label = int(df_p["Class_numeric"].iloc[0])
    wins = []
    for start in range(0, max(len(data) - window_size + 1, 0), step):
        w = data[start : start + window_size]
        if w.shape[0] == window_size and np.isfinite(w).all():
            wins.append(w)
    return wins, [label] * len(wins)


def build_xy(parts):
    X, y = [], []
    for p in parts:
        df_p = df[df["Participant"] == p]
        wins, labs = generate_windows(df_p)
        X.extend(wins)
        y.extend(labs)
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    if len(X):
        m = mask_valid_windows(X)
        X, y = X[m], y[m]
    return X, y


X_train, y_train = build_xy(train_parts_final)
X_val, y_val = build_xy(val_parts)
X_test, y_test = build_xy(test_parts)

log(f"Janelas: train={len(X_train)} | val={len(X_val)} | test={len(X_test)}")
if min(len(X_train), len(X_val), len(X_test)) == 0:
    raise RuntimeError(
        "Conjunto vazio após janelação/filtragem. Verifique preprocessamento."
    )


# Checagem: não permitir classe única em nenhum split de janelas
def check_binary(y, split_name):
    uniq = np.unique(y).tolist()
    if not set(uniq).issubset({0, 1}):
        raise RuntimeError(f"{split_name}: labels inválidos {uniq}")
    if len(uniq) < 2:
        raise RuntimeError(
            f"{split_name}: apenas uma classe presente (labels={uniq}). "
            "Isso distorce métricas. Ajuste os participantes ou parâmetros."
        )


check_binary(y_train, "TRAIN(windows)")
check_binary(y_val, "VAL(windows)")
check_binary(y_test, "TEST(windows)")

# ---------------------- Normalização (stats do treino) ----------------------
means = np.mean(X_train, axis=0, dtype=np.float64)
stds = np.std(X_train, axis=0, dtype=np.float64)
stds[stds == 0] = 1.0


def norm(X):
    return ((X - means) / stds).astype(np.float32)


X_train = norm(X_train)
X_val = norm(X_val)
X_test = norm(X_test)

# salvar normalização
np.savez(os.path.join(OUT_DIR, "normalization_stats_debug.npz"), means=means, stds=stds)

# ---------------------- Datasets & Loaders ----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = TensorDataset(
    torch.from_numpy(X_train), torch.from_numpy(y_train).float().unsqueeze(1)
)
val_ds = TensorDataset(
    torch.from_numpy(X_val), torch.from_numpy(y_val).float().unsqueeze(1)
)
test_ds = TensorDataset(
    torch.from_numpy(X_test), torch.from_numpy(y_test).float().unsqueeze(1)
)

# Sampler balanceado no treino (inverso da frequência)
class_counts = np.bincount(y_train.astype(int), minlength=2)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[y_train.astype(int)]
sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False)


# ---------------------- Modelo ----------------------
class BiLSTM(nn.Module):
    def __init__(self, input_size=len(features), hidden=96, drop=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(2 * hidden, 1)

    def forward(self, x):
        self.lstm.flatten_parameters()
        x, _ = self.lstm(x)  # (B,T,2H)
        x = x[:, -1, :]  # último passo
        x = self.drop(x)
        return self.fc(x)  # logits


model = BiLSTM().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))

# ---------------------- Treino (early stopping) ----------------------
best_val = float("inf")
best_state = None
bad = 0
train_losses, val_losses = [], []

for ep in range(1, EPOCHS + 1):
    model.train()
    tr = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        tr += loss.item()
    tr /= max(1, len(train_loader))
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            vl += criterion(model(xb), yb).item()
    vl /= max(1, len(val_loader))
    val_losses.append(vl)

    print(f"Epoch {ep:02d} | train_loss={tr:.4f} | val_loss={vl:.4f}")

    if vl < best_val - 1e-4:
        best_val = vl
        best_state = {
            k: v.detach().cpu().clone() for k, v in model.state_dict().items()
        }
        bad = 0
    else:
        bad += 1
        if bad >= PATIENCE:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

# salvar curvas
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves_bilstm_debug.png"), dpi=150)
plt.close()

# salvar pesos
torch.save(model.state_dict(), os.path.join(OUT_DIR, "bilstm_debug_best.pth"))


# ---------------------- Seleção de THRESHOLD (nível SUJEITO) ----------------------
def subject_val_records(parts, model, means, stds, device):
    recs = []
    model.eval()
    with torch.no_grad():
        for p in parts:
            df_p = df[df["Participant"] == p]
            wins, labs = generate_windows(df_p)
            if len(wins) == 0:
                continue
            Xp = ((np.array(wins, dtype=np.float32) - means) / stds).astype(np.float32)
            probs = (
                torch.sigmoid(model(torch.from_numpy(Xp).to(device)))
                .cpu()
                .numpy()
                .ravel()
            )
            y = int(labs[0])
            recs.append((p, y, probs))
    return recs


val_recs = subject_val_records(val_parts, model, means, stds, device)

best_th, best_bacc = 0.5, -1
for t in THRESH_GRID:
    y_true, y_pred = [], []
    for _, y, pr in val_recs:
        m = trimmed_mean(pr, TRIM)
        y_true.append(y)
        y_pred.append(int(m > t))
    if y_true:
        bacc = balanced_accuracy_score(y_true, y_pred)
        if bacc > best_bacc:
            best_bacc, best_th = bacc, t

print(f"Threshold ótimo (VAL sujeito): {best_th:.2f} | bAcc={best_bacc:.3f}")
save_json(
    os.path.join(OUT_DIR, "threshold_debug.json"),
    {"best_threshold": float(best_th), "balanced_accuracy_val": float(best_bacc)},
)


# ---------------------- Avaliação por janelas ----------------------
def eval_windows(loader, th):
    probs, labels = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            probs.append(torch.sigmoid(model(xb)).cpu().numpy())
            labels.append(yb.numpy())
    probs = np.vstack(probs).ravel()
    labels = np.vstack(labels).ravel().astype(int)
    preds = (probs > th).astype(int)
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    return acc, prec, rec, f1, cm, labels, preds, probs


acc_w, p_w, r_w, f1_w, cm_w, y_w, yhat_w, pr_w = eval_windows(test_loader, best_th)
print(
    f"[Janelas - Test BiLSTM DEBUG] Acc={acc_w*100:.2f}% | Precision={p_w:.3f} | Recall={r_w:.3f} | F1={f1_w:.3f}"
)
plot_confusion(
    cm_w,
    "Confusion Matrix (Windows) - BiLSTM DEBUG",
    os.path.join(OUT_DIR, "confusion_matrix_windows_bilstm_debug.png"),
)

# === ROC, PR e Calibração (Windows) + dump de saídas ===
# ROC / AUC
fpr, tpr, _ = roc_curve(y_w, pr_w)
auc_roc = roc_auc_score(y_w, pr_w)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC={auc_roc:.3f}")
plt.plot([0, 1], [0, 1], "--", lw=1)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC - BiLSTM DEBUG (Windows)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_windows_bilstm_debug.png"), dpi=150)
plt.close()

# PR / AP
prec_curve, rec_curve, _ = precision_recall_curve(y_w, pr_w)
ap = average_precision_score(y_w, pr_w)
plt.figure(figsize=(5, 4))
plt.plot(rec_curve, prec_curve, label=f"AP={ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall - BiLSTM DEBUG (Windows)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pr_windows_bilstm_debug.png"), dpi=150)
plt.close()

# Calibração (reliability curve)
prob_true, prob_pred = calibration_curve(y_w, pr_w, n_bins=10, strategy="quantile")
plt.figure(figsize=(5, 4))
plt.plot(prob_pred, prob_true, marker="o")
plt.plot([0, 1], [0, 1], "--", lw=1)
plt.xlabel("Predicted prob")
plt.ylabel("Observed freq")
plt.title("Calibration - BiLSTM DEBUG (Windows)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "calibration_windows_bilstm_debug.png"), dpi=150)
plt.close()

# Dumps para reprodutibilidade
np.savez(
    os.path.join(OUT_DIR, "test_windows_outputs.npz"),
    probs=pr_w.astype(np.float32),
    preds=yhat_w.astype(np.int8),
    labels=y_w.astype(np.int8),
    threshold=np.array([best_th], dtype=np.float32),
)

# (Opcional) CSV com nível de janela — bom para auditoria e tabelas
pd.DataFrame(
    {"label": y_w.astype(int), "pred": yhat_w.astype(int), "prob": pr_w.astype(float)}
).to_csv(os.path.join(OUT_DIR, "windows_predictions.csv"), index=False)

# Salvar métricas de janelas (incluindo AUC/AP/threshold)
win_json = {
    "accuracy": float(acc_w),
    "precision": float(p_w),
    "recall": float(r_w),
    "f1": float(f1_w),
    "auc_roc": float(auc_roc),
    "average_precision": float(ap),
    "threshold_used": float(best_th),
}
save_json(os.path.join(OUT_DIR, "metrics_windows.json"), win_json)


# ---------------------- Avaliação por SUJEITO ----------------------
def participant_vote(parts, th, agg="trimmed_mean"):
    y_true, y_pred, mean_probs, pids = [], [], [], []
    model.eval()
    with torch.no_grad():
        for p in parts:
            df_p = df[df["Participant"] == p]
            wins, labs = generate_windows(df_p)
            if len(wins) == 0:
                continue
            Xp = ((np.array(wins, dtype=np.float32) - means) / stds).astype(np.float32)
            probs = (
                torch.sigmoid(model(torch.from_numpy(Xp).to(device)))
                .cpu()
                .numpy()
                .ravel()
            )
            y_true.append(int(labs[0]))
            mprob = (
                trimmed_mean(probs, TRIM)
                if agg == "trimmed_mean"
                else float((probs > th).mean())
            )
            mean_probs.append(mprob)
            pids.append(int(p))
            y_pred.append(int(mprob > th))
    return np.array(y_true), np.array(y_pred), np.array(mean_probs), np.array(pids)


y_true_s, y_pred_s, mprob_s, pids_s = participant_vote(
    test_parts, best_th, agg="trimmed_mean"
)
acc_s = accuracy_score(y_true_s, y_pred_s)
prec_s = precision_score(y_true_s, y_pred_s, zero_division=0)
rec_s = recall_score(y_true_s, y_pred_s, zero_division=0)
f1_s = f1_score(y_true_s, y_pred_s, zero_division=0)

print(f"[Participantes - Test BiLSTM DEBUG] Acc={acc_s*100:.2f}% (n={len(y_true_s)})")

# Confusão por sujeitos
cm_s = confusion_matrix(y_true_s, y_pred_s, labels=[0, 1])
plot_confusion(
    cm_s,
    "Confusion Matrix (Subjects) - BiLSTM DEBUG",
    os.path.join(OUT_DIR, "confusion_matrix_subjects_bilstm_debug.png"),
)

# salvar métricas e CSV por sujeito
save_json(
    os.path.join(OUT_DIR, "metrics_subjects.json"),
    {
        "accuracy": float(acc_s),
        "precision": float(prec_s),
        "recall": float(rec_s),
        "f1": float(f1_s),
        "n_subjects_test": int(len(y_true_s)),
    },
)

subj_df = pd.DataFrame(
    {
        "Participant": pids_s,
        "TrueClass": y_true_s.astype(int),
        "PredClass": y_pred_s.astype(int),
        "MeanProbASD": mprob_s.astype(float),
        "ThresholdUsed": np.full_like(mprob_s, fill_value=best_th, dtype=float),
    }
).sort_values("Participant")
subj_df["Correct"] = (subj_df["TrueClass"] == subj_df["PredClass"]).astype(int)
subj_df.to_csv(os.path.join(OUT_DIR, "subjects_predictions.csv"), index=False)

# ---------------------- Resumo final ----------------------
print("\n✅ Artefatos salvos em:", OUT_DIR)
print(" - bilstm_debug_best.pth")
print(" - normalization_stats_debug.npz")
print(" - splits_debug.json")
print(" - threshold_debug.json")
print(" - training_curves_bilstm_debug.png")
print(" - confusion_matrix_windows_bilstm_debug.png")
print(" - confusion_matrix_subjects_bilstm_debug.png")
print(" - roc_windows_bilstm_debug.png")
print(" - pr_windows_bilstm_debug.png")
print(" - calibration_windows_bilstm_debug.png")
print(" - metrics_windows.json")
print(" - metrics_subjects.json")
print(" - windows_predictions.csv")
print(" - test_windows_outputs.npz")

# sujeitos: total=57 | train=36 | val=9 | test=12
Janelas: train=4510 | val=1132 | test=1778
Epoch 01 | train_loss=0.4700 | val_loss=1.2080
Epoch 02 | train_loss=0.1793 | val_loss=1.1946
Epoch 03 | train_loss=0.1027 | val_loss=1.0663
Epoch 04 | train_loss=0.1537 | val_loss=1.3319
Epoch 05 | train_loss=0.1390 | val_loss=1.0663
Epoch 06 | train_loss=0.0803 | val_loss=1.3785
Epoch 07 | train_loss=0.0624 | val_loss=1.3683
Early stopping.
Threshold ótimo (VAL sujeito): 0.18 | bAcc=0.875
[Janelas - Test BiLSTM DEBUG] Acc=80.65% | Precision=0.903 | Recall=0.630 | F1=0.743
[Participantes - Test BiLSTM DEBUG] Acc=75.00% (n=12)

✅ Artefatos salvos em: results_bilstm_debug_20260714-124249
 - bilstm_debug_best.pth
 - normalization_stats_debug.npz
 - splits_debug.json
 - threshold_debug.json
 - training_curves_bilstm_debug.png
 - confusion_matrix_windows_bilstm_debug.png
 - confusion_matrix_subjects_bilstm_debug.png
 - roc_windows_bilstm_debug.png
 - pr_windows_bilstm_debug.png
 - calibration_windo